# Notebook 03 — Prescriptive Optimization
**MSE 433 — Olympic Medal Performance**

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.optimization import (
    compute_sport_params, adjust_r_for_profile,
    solve_sport_allocation, budget_sensitivity,
    r_sensitivity, load_zinb
)

ROOT   = Path('..').resolve()
INPUT  = ROOT / 'data' / 'input'
OUTPUT = ROOT / 'data' / 'output'
FIG    = ROOT / 'report' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete')

## 1. Sport-Level Parameters

In [ ]:
sport_params_summer = compute_sport_params(season='Summer', min_sport_obs=5)

print(f'Summer sports with sufficient history: {len(sport_params_summer)}')
print('\nTop 15 sports by historical medal rate per athlete:')
display(sport_params_summer[['Sport', 'r_base', 'm_s', 'M_s', 'n_obs']].head(15).round(4))
print('\nr_base  = avg medals won per athlete in that sport (historical 1960-2016)')
print('m_s     = minimum athletes to enter (10th percentile of historical participation)')
print('M_s     = IOC quota ceiling (90th percentile of historical participation)')

## 2. Load ZINB Model & Define NSA Profiles

In [ ]:
zinb_summer = load_zinb('Summer')
print('ZINB model loaded.')

# ── General Framework: Two Representative NSA Profiles ────────────────────────
# Profile A: Mid-Tier Nation (~median GDP, ~100 athletes)
profile_A = {
    'name':              'Mid-Tier Nation',
    'gdp_per_capita':    15_000,
    'population':        30_000_000,
    'female_ratio':      0.40,
    'is_host':           0,
    'delegation_budget': 100,
}

# Profile B: Small Emerging Nation (low GDP, small delegation)
profile_B = {
    'name':              'Small Emerging Nation',
    'gdp_per_capita':    3_000,
    'population':        8_000_000,
    'female_ratio':      0.20,
    'is_host':           0,
    'delegation_budget': 30,
}

# ── Case Study: Canada at Rio 2016 ───────────────────────────────────────────
# Source: Canadian Olympic Committee (2016); Statistics Canada
# 314 athletes across 27 sports; Own the Podium cycle budget CAD $37M
# female_ratio = 186/314 = 0.592 (highest in Canadian Olympic history)
# GDP per capita ~USD $42,000 (World Bank, 2016); population ~36M
profile_canada = {
    'name':              'Canada — Rio 2016',
    'gdp_per_capita':    42_000,
    'population':        36_000_000,
    'female_ratio':      0.592,
    'is_host':           0,
    'delegation_budget': 314,
}

for p in [profile_A, profile_B, profile_canada]:
    print(f"  {p['name']}: {p['delegation_budget']} athletes, "
          f"GDP/cap=${p['gdp_per_capita']:,}, "
          f"female_ratio={p['female_ratio']:.2f}")

## 3. Adjust Sport Medal Rates for Each Profile

In [ ]:
# The ZINB coefficients scale r_base up or down based on how each NSA's
# profile (GDP, population, female_ratio, is_host) differs from the median country.
sp_A      = adjust_r_for_profile(sport_params_summer, zinb_summer, profile_A)
sp_B      = adjust_r_for_profile(sport_params_summer, zinb_summer, profile_B)
sp_canada = adjust_r_for_profile(sport_params_summer, zinb_summer, profile_canada)

print(f"Profile A      adjustment factor: {sp_A['adj_factor'].iloc[0]:.4f}  "
      f"({'above' if sp_A['adj_factor'].iloc[0] > 1 else 'below'} average)")
print(f"Profile B      adjustment factor: {sp_B['adj_factor'].iloc[0]:.4f}  "
      f"({'above' if sp_B['adj_factor'].iloc[0] > 1 else 'below'} average)")
print(f"Canada 2016    adjustment factor: {sp_canada['adj_factor'].iloc[0]:.4f}  "
      f"({'above' if sp_canada['adj_factor'].iloc[0] > 1 else 'below'} average)")

print('\nTop 10 sports by Canada-adjusted medal rate:')
display(sp_canada[['Sport', 'r_base', 'r_adjusted', 'm_s', 'M_s']].head(10).round(4))

## 4. Solve ILP — General Framework (Profiles A & B)

In [ ]:
result_A = solve_sport_allocation(sp_A, profile_A['delegation_budget'],
                                  r_col='r_adjusted', top_n_sports=40, verbose=False)
result_B = solve_sport_allocation(sp_B, profile_B['delegation_budget'],
                                  r_col='r_adjusted', top_n_sports=40, verbose=False)

for label, result, profile in [('Profile A — Mid-Tier Nation', result_A, profile_A),
                                 ('Profile B — Small Nation',   result_B, profile_B)]:
    print(f'\n=== {label} ===')
    print(f'  Status:           {result["status"]}')
    print(f'  Expected medals:  {result["objective"]:.2f}')
    print(f'  Athletes used:    {result["total_athletes"]} / {profile["delegation_budget"]}')
    print(f'  Sports selected:  {len(result["allocation"])}')
    if result['shadow_price'] is not None:
        print(f'  Shadow price:     {result["shadow_price"]:.4f} medals per extra athlete')
        print(f'  → 10 more athletes ≈ +{result["shadow_price"]*10:.2f} expected medals')
    print(f'\n  Allocation:')
    display(result['allocation'][['Sport', 'athletes_allocated', 'expected_medals']].round(3))

## 5. Case Study — Canada at Rio 2016

In [ ]:
print('=== Canada 2016 — Optimal Allocation (Gurobi ILP) ===')
result_canada = solve_sport_allocation(sp_canada, profile_canada['delegation_budget'],
                                       r_col='r_adjusted', top_n_sports=40, verbose=False)

print(f'Status:           {result_canada["status"]}')
print(f'Expected medals:  {result_canada["objective"]:.2f}')
print(f'Athletes used:    {result_canada["total_athletes"]} / {profile_canada["delegation_budget"]}')
print(f'Sports selected:  {len(result_canada["allocation"])}')
if result_canada['shadow_price'] is not None:
    print(f'Shadow price:     {result_canada["shadow_price"]:.4f} medals per extra athlete')
    print(f'  → Each additional 10 athletes ≈ +{result_canada["shadow_price"]*10:.2f} expected medals')
    print(f'  → KB (F): Marginal medal return = shadow price on delegation budget constraint')

print('\n=== Optimal Sport Portfolio — Canada 2016 ===')
display(result_canada['allocation'][['Sport', 'athletes_allocated', 'expected_medals']].round(3))

# Actual Canada 2016 result for comparison
print('\n=== Actual Canada 2016 Result ===')
print('Athletes sent:  314 across 27 sports')
print('Medals won:     22 (4 gold, 3 silver, 15 bronze)')
print('Ranking:        10th overall (best non-boycotted Summer result)')
print(f'\nModel expected: {result_canada["objective"]:.2f} medals')
print(f'Actual:         22 medals')
gap = result_canada["objective"] - 22
print(f'Gap:            {gap:+.2f} (positive = model predicts more than actual — opportunity cost of suboptimal allocation)')

## Figure 11 — Sport Allocation Bar Charts (All Three Profiles)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

profiles_results = [
    ('Profile A\n(Mid-Tier, 100 athletes)', result_A,      '#2196F3'),
    ('Profile B\n(Small Nation, 30 athletes)', result_B,   '#FF5722'),
    ('Canada — Rio 2016\n(314 athletes)',  result_canada,  '#4CAF50'),
]

for ax, (label, result, color) in zip(axes, profiles_results):
    alloc = result['allocation'].sort_values('expected_medals', ascending=True)
    if len(alloc) == 0:
        ax.text(0.5, 0.5, 'No feasible allocation', ha='center', va='center',
                transform=ax.transAxes)
        continue
    y = range(len(alloc))
    ax.barh(y, alloc['expected_medals'], color=color, alpha=0.8)
    ax.set_yticks(list(y))
    ax.set_yticklabels(
        [f"{row['Sport']} ({int(row['athletes_allocated'])})"
         for _, row in alloc.iterrows()],
        fontsize=8
    )
    ax.set_xlabel('Expected Medals')
    ax.set_title(f'{label}\nTotal expected: {result["objective"]:.1f} medals',
                 fontsize=10)

fig.suptitle('Figure 11: Optimal Sport Portfolio — Gurobi ILP Results\n'
             'Numbers in parentheses = athletes allocated to that sport\n'
             'KB: (A) Resource Allocation, Integer Programming, Gurobi',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig11_sport_allocation.png', bbox_inches='tight')
plt.show()
print('Saved Figure 11')

## Figure 12 — Budget Sensitivity (Canada & Profile A)

In [ ]:
print('Running budget sensitivity for Canada and Profile A...')
budgets_canada = list(range(50, 401, 20))
budgets_A      = list(range(10, 201, 10))

sens_canada = budget_sensitivity(sp_canada, budgets_canada, r_col='r_adjusted', top_n_sports=40)
sens_A      = budget_sensitivity(sp_A,      budgets_A,      r_col='r_adjusted', top_n_sports=40)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, sens, budget_actual, color, marker_label) in zip(axes, [
    ('Profile A — Mid-Tier Nation', sens_A,      100, '#2196F3', 'Current budget = 100'),
    ('Canada — Rio 2016',           sens_canada, 314, '#4CAF50', 'Actual delegation = 314'),
]):
    valid = sens.dropna(subset=['expected_medals'])
    ax.plot(valid['delegation_budget'], valid['expected_medals'],
            color=color, lw=2.5, marker='o', markersize=4)
    ax.axvline(budget_actual, color='grey', linestyle='--', lw=1.5, label=marker_label)

    # Shadow price annotation at actual budget
    at_budget = valid[valid['delegation_budget'] == budget_actual]
    if not at_budget.empty and at_budget['shadow_price'].notna().any():
        sp = at_budget['shadow_price'].iloc[0]
        ax.annotate(f'Shadow price ≈ {sp:.3f}\nmedals/athlete',
                    xy=(budget_actual, at_budget['expected_medals'].iloc[0]),
                    xytext=(budget_actual + 20, at_budget['expected_medals'].iloc[0] * 0.85),
                    fontsize=9, arrowprops=dict(arrowstyle='->', color='black'))

    ax.set_xlabel('Delegation Budget (athletes)')
    ax.set_ylabel('Expected Total Medals')
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=9)

fig.suptitle('Figure 12: Budget Sensitivity — Expected Medals vs. Delegation Size\n'
             'Slope of curve = marginal medal return (shadow price) per additional athlete\n'
             'KB: (A) Sensitivity Analysis; (F) Marginal Cost Analysis',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig12_budget_sensitivity.png', bbox_inches='tight')
plt.show()
print('Saved Figure 12')

print('\nCanada budget sensitivity table (selected rows):')
display(sens_canada[sens_canada['delegation_budget'].isin([100,150,200,250,314,350,400])])

## Table — r_s Parameter Sensitivity (Canada — Solution Robustness)

In [ ]:
# Vary all r_s by ±20% to test if the optimal sport portfolio is robust
# to uncertainty in the medal rate estimates.
print('=== r_s Sensitivity — Canada 2016 ===')
print('If expected medals change proportionally with r_s, the solution is structurally stable.')
rsens_canada = r_sensitivity(sp_canada, profile_canada['delegation_budget'],
                             deltas=[-0.20, -0.10, 0, 0.10, 0.20],
                             r_col='r_adjusted', top_n_sports=40)
display(rsens_canada)

pct_change = ((rsens_canada['expected_medals'].iloc[-1] - rsens_canada['expected_medals'].iloc[0])
              / rsens_canada['expected_medals'].iloc[2] * 100)
print(f'\n±20% swing in all r_s → {pct_change:.1f}% swing in expected medals')
print('A proportional response indicates the solution is well-structured (not a knife-edge solution).')

## Key Findings

In [ ]:
print('=== OPTIMIZATION FINDINGS ===')

for label, result, profile in [
    ('Profile A — Mid-Tier',  result_A,      profile_A),
    ('Profile B — Small',     result_B,      profile_B),
    ('Canada 2016',           result_canada, profile_canada),
]:
    alloc = result.get('allocation', pd.DataFrame())
    print(f'\n{label}:')
    print(f'  Budget: {profile["delegation_budget"]} athletes')
    print(f'  Optimal expected medals: {result.get("objective", "N/A"):.2f}')
    print(f'  Sports selected: {len(alloc)}')
    if result.get('shadow_price') is not None:
        print(f'  Shadow price: {result["shadow_price"]:.4f} medals/athlete')
        print(f'  OTP interpretation: +10 athletes → +{result["shadow_price"]*10:.2f} expected medals')
    if len(alloc) > 0:
        top3 = alloc.head(3)
        print(f'  Top 3 sports: {", ".join(top3["Sport"].tolist())}')

print('\n=== Canada Comparison ===')
print(f'Optimal expected (model): {result_canada["objective"]:.2f}')
print(f'Actual result (2016):     22')
print(f'Sports: optimal={len(result_canada["allocation"])} vs actual=27')